# Static Topology Evaluation and Invariant Verification

## Install

```bash
pip install graphyco
```

---

## Environment and Imports

Import dependencies from core and bridge.

In [ ]:
import sys, os
for p in [os.path.abspath("../../validation"), os.path.abspath("../validation"), os.path.abspath("validation")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
import sys, os
import torch
import torch.nn as nn
import pandas as pd

from graphyco.bridge.torch_bridge import evaluate_model, trace_to_graph
from graphyco.core.invariants import validate_invariants, InvariantViolationError
from graphyco.core.primitives import SCALE

torch.manual_seed(42)
print(f"PyTorch Version: {torch.__version__}")

## Architecture Definitions

Define a Sequential MLP and a ResNet block to compare linear vs. residual topologies.

In [ ]:
class SequentialMLP(nn.Module):
    def __init__(self, in_dim=32, hidden_dim=64, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, x):
        return self.net(x)

class ResNetBlock(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.l1 = nn.Linear(dim, dim)
        self.r1 = nn.ReLU()
        self.l2 = nn.Linear(dim, dim)
        self.r2 = nn.ReLU()
        self.out = nn.Linear(dim, 10)
    def forward(self, x):
        h = self.l2(self.r1(self.l1(x)))
        return self.out(self.r2(h + x))

mlp = SequentialMLP()
resnet = ResNetBlock()
print("Models initialized successfully.")

## Static Graph Extraction & Topological Metrics

Extract operational graphs via PyTorch FX tracing and compute deterministic invariants.

In [ ]:
res_mlp = evaluate_model(mlp, mode="trace")
res_resnet = evaluate_model(resnet, mode="trace")

ev_mlp = res_mlp["evaluation"]
ev_res = res_resnet["evaluation"]

table = [
    {
        "Architecture": "Sequential MLP",
        "Nodes": ev_mlp["node_count"],
        "Edges": ev_mlp["edge_count"],
        "Density": ev_mlp["graph_density"] / SCALE,
        "Coherence": ev_mlp["connectivity_coherence"] / SCALE,
        "Bottleneck Ratio": ev_mlp["topological_bottleneck_ratio"] / SCALE,
        "Resilience": ev_mlp["structural_perturbation_resilience"] / SCALE,
    },
    {
        "Architecture": "ResNet Block",
        "Nodes": ev_res["node_count"],
        "Edges": ev_res["edge_count"],
        "Density": ev_res["graph_density"] / SCALE,
        "Coherence": ev_res["connectivity_coherence"] / SCALE,
        "Bottleneck Ratio": ev_res["topological_bottleneck_ratio"] / SCALE,
        "Resilience": ev_res["structural_perturbation_resilience"] / SCALE,
    }
]
pd.DataFrame(table)

## Graph Invariant Verification

Validate graph integrity against the 7 core invariants on the module graph and observe port checks.

In [ ]:
# Verify invariants on module-level graph representation
res_mod = evaluate_model(resnet, mode="module")
validate_invariants(res_mod["graph_state"])
print("Module-level graph invariants verified successfully.")
print(f"  - Active Nodes: {len(res_mod['graph_state'].nodes)}")
print(f"  - Total Edges : {len(res_mod['graph_state'].edges)}")

# Inspect port connectivity invariant behavior on open FX DAGs
try:
    validate_invariants(res_resnet["graph_state"])
except InvariantViolationError as ex:
    print(f"Captured expected invariant violation on open terminal port: {ex}")

## Perturbation Cascade Profile

Inspect cascade failure costs across nodes in the ResNet architecture.

In [ ]:
profile = ev_res["perturbation_profile"]
top_fragile = sorted(profile.items(), key=lambda x: -x[1])[:5]

print("Top 5 Fragile Nodes (Cascade Perturbation Cost):")
for nid, cost in top_fragile:
    print(f"  {nid:<16}: cost = {cost}")

## Width-Scale Invariance

Confirm that scaling layer width does not alter topological invariants in feedforward chains.

In [ ]:
mlp_small = SequentialMLP(hidden_dim=32)
mlp_large = SequentialMLP(hidden_dim=512)

ev_s = evaluate_model(mlp_small, mode="trace")["evaluation"]
ev_l = evaluate_model(mlp_large, mode="trace")["evaluation"]

params_s = sum(p.numel() for p in mlp_small.parameters())
params_l = sum(p.numel() for p in mlp_large.parameters())

print(f"Small MLP: {params_s:,} parameters | Bottleneck: {ev_s['topological_bottleneck_ratio'] / SCALE:.4f}")
print(f"Large MLP: {params_l:,} parameters | Bottleneck: {ev_l['topological_bottleneck_ratio'] / SCALE:.4f}")
assert ev_s["topological_bottleneck_ratio"] == ev_l["topological_bottleneck_ratio"]
print("Width-scale invariance verified.")